# OPTIONAL Workbook for Final Project, Part 3

Your group is welcome to work on a local version of a notebook for this assignment.

This workspace is here if you'd rather not have to install all necessary packages locally.

You can download any json files to your local computer to add them to your local jekyll page.

# Group Members:

## Shane Ghosh, Shrey Ghosh (Partner for Part 2 and 3)

## Dataset URLs  
You can obtain our main dataset from the URL given here: https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.CD

You can obtain our contextual dataset from the URL given here: https://data.worldbank.org/indicator/SL.TLF.ACTI.ZS

In [ ]:
import ipywidgets as widgets
from ipywidgets.embed import embed_minimal_html
import numpy as np
import pandas as pd
import zipfile
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import altair as alt

#: :generated:
#: :model: ChatGPT5.2
#: :prompt: this is how I am doing it downloading the data locally, please do exactly the same thing but without downloading the data 
#: :changes:
#: :response:
url = "https://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.PP.CD?downloadformat=csv"

response = requests.get(url)
z = zipfile.ZipFile(BytesIO(response.content))

data_file = [f for f in z.namelist() if f.startswith("API_") and f.endswith(".csv")][0]

df = pd.read_csv(
    z.open(data_file),
    skiprows=4
)

#: :generated:
#: :model: ChatGPT5.2
#: :prompt: how can I filter the data to only take between 1990 and 2024 (drop other columns)  
#: :changes: Dropped columns (years) which were less than 1990 or greater than 2024
#: :response:
year_cols = df.columns[4:]

cols_to_keep = ["Country Name", "Country Code"] + [
    col for col in df.columns[4:]
    if col.isdigit() and 1990 <= int(col) <= 2024
]

df_filtered = df[cols_to_keep]

url_labor = "https://api.worldbank.org/v2/en/indicator/SL.TLF.ACTI.ZS?downloadformat=csv"

response_labor = requests.get(url_labor)
z_labor = zipfile.ZipFile(BytesIO(response_labor.content))

data_file_labor = [
    f for f in z_labor.namelist()
    if f.startswith("API_") and f.endswith(".csv")
][0]

df_labor = pd.read_csv(
    z_labor.open(data_file_labor),
    skiprows=4
)

cols_to_keep_labor = ["Country Name", "Country Code"] + [
    col for col in df_labor.columns[4:]
    if col.isdigit() and 1990 <= int(col) <= 2024
]

df_labor_filtered = df_labor[cols_to_keep_labor]

In [ ]:
year_cols = [col for col in df_filtered.columns if col.strip().isdigit()]
default_year = '2015'

#: :generated:
#: :model: Claude
#: :prompt: convert my bqplot dashboard to altair with minimal changes. 
#:          I had a bar chart of top 20 countries by GDP linked to a line 
#:          chart showing GDP over time when a bar is clicked
#: :changes: 
#: :response:
top_df = df[['Country Name', default_year]].dropna().copy()
top_df[default_year] = pd.to_numeric(top_df[default_year], errors='coerce')
top_df = top_df.dropna().sort_values(default_year, ascending=False).head(20)

top_countries = top_df['Country Name'].tolist()
line_df = df_filtered[df_filtered['Country Name'].isin(top_countries)].melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=year_cols,
    var_name='Year',
    value_name='GDP per Capita'
)
line_df['GDP per Capita'] = pd.to_numeric(line_df['GDP per Capita'], errors='coerce')

selection = alt.selection_point(fields=['Country Name'])

bars = (
    alt.Chart(top_df)
    .mark_bar()
    .encode(
        x=alt.X('Country Name:N', sort='-y', title='Country',
                axis=alt.Axis(labelAngle=-45)),
        y=alt.Y(f'{default_year}:Q', title='GDP per Capita'),
        color=alt.condition(selection, alt.value('steelblue'), alt.value('lightgray')),
        tooltip=['Country Name:N', f'{default_year}:Q']
    )
    .add_params(selection)
    .properties(
        title=f'Top 20 Countries by GDP per Capita ({default_year})',
        width=500,
        height=350
    )
)

lines = (
    alt.Chart(line_df)
    .mark_line(point=True)
    .encode(
        x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('GDP per Capita:Q', title='GDP per Capita'),
        color=alt.Color('Country Name:N', legend=None),
        tooltip=['Country Name:N', 'Year:O', 'GDP per Capita:Q']
    )
    .transform_filter(selection)
    .properties(
        title='Click a country to see its GDP over time',
        width=500,
        height=350
    )
)

dashboard = bars | lines
dashboard

The driver plot is a bar graph which shows the GDP per capita of the 20 countries in the dataset with the highest GDP per capita for the year 2015. Since this data is not continuous and rather
  is categorical in nature, we have decided to use a bar plot to depict it. As for color, we have decided to use the steel blue color for all bars, and we have decided to change all other bars
  except the selected bar to light gray once a bar is selected. This colormap allows the user to easily decipher when no country is selected, and when a country gets selected. There was no need
  to apply different colors to the bars when unselected because the x-axis adequately shows the user which country corresponds to which bar. Additionally, when hovering over the bar, the country
  name and the GDP per capita for the 2015 year is visible to the user for ease of use.
  
  The driven plot is a line graph which shows the GDP per capita between the years 1990 and 2024 for all countries if no country is selected on the driver plot, or a particular country
  corresponding to the driver plot selection. Since this data is continous, as this is for GDP per capita over time, we felt that the use of a line graph would be most appropriate, as
  interpolation of the data is appropriate here and the continous nature of the data fits a line graph. Furthermore, when unselected, we decided to use different colors to represent the different
  countries, as can be seen in the plot when no country is selected for the corresponding driver plot. We decided not to include a legend for these colors, as the user can hover over these lines
  to get information about the country corresponding to that particular line as well as the GDP per Capita for that year mark. Due to this functionality, we felt that a legend would unnecessarily
  clutter up the visualization, without much additional benefit to the plots. Therefore, we felt that having each country represented by a different color was a sufficient distinction of different
  countries in this graph.  Additionally, when a country is selected in the driver plot, the color of the line for the selected country changes to steel blue, since only one country's data is
  displayed when a selection is made on the driver plot, removing the need for a color map.
  
  When looking at the data, one common trend we notice is that economies with many industries and many sectors fueling the economy, such as that of the United States, North America as a
  continent, and the Netherlands, have had much more stable and controlled growth in GDP per capita as compared to countries such as Qatar or the UAE. The reason for this is because, unlike
  the countries previously mentioned, countries like Qatar and the UAE usually rely on a single source for most of their GDP, oil in the case of these two countries. Therefore, when this resource
  goes down in demand, such as what happened during the oil crash of the 90's as well as the 2014 oil crash, the GDP per capita of these economies fall dramatically. On the other end, when resources
  such as oil have their demand outpace their supply, such as what happened during the early 2010's and up to the pandemic with oil, then we see a massive spike in GDP per capita for oil dependent
  nations such as Qatar and the UAE. This leads to more erratic and volatile GDP per capita graphs for these countries compared with countries such as the US or Netherlands.

In [ ]:
common_years = sorted(set(df_filtered.columns[4:]) & set(df_labor_filtered.columns[4:]))
gdp = df_filtered.set_index("Country Name")[common_years]
labor = df_labor_filtered.set_index("Country Name")[common_years]

scatter = pd.DataFrame({"Country": gdp.index, "GDP": gdp["2015"].values, "Labor": labor["2015"].values}).dropna()

plt.figure(figsize=(10, 6))

plt.scatter(
    scatter["Labor"],
    scatter["GDP"]
)

plt.xlabel("Labor Force Participation Rate (%)")
plt.ylabel("GDP per capita (PPP)")
plt.title(f"GDP vs Labor Force Participation 2015")

plt.tight_layout()
plt.savefig("plot2.png")
plt.show()

This graph shows GDP per Capita (PPP) vs. Labor Force Participation Rate for all countries in the dataset in the year 2015. The scatter plot shows that there is generally a positive relationship between labor force participation rate and GDP per Capita (PPP). This scatter plot also serves well as a snapshot of how the world is doing. In general, countries with a high labor force participation rate but low GDP per Capita mean that those countries have inefficient economies.

If a country has lower than average GDP per Capita, we can distinguish it into two groups. The first group is if the country has a lower than 65% labor force participation rate. If so, then the country needs to work on increasing its labor force participation rate, as its economy can grow from employing more people and using the labor resources at its disposal. These countries are normally the ones most affected by war, extreme poverty, and crime, as these are structural problems that stop major parts of the population from being able to work. The next group of countries is those with a labor force participation rate greater than 65%. This means that the country is on its way to improvement, as most of its population has been able to participate in the workforce and does not need to practice subsistence farming. These countries have started their transition toward becoming high-income, export-oriented economies but are currently only producing cheap, low-profit materials. Once these countries graduate from making cheap, low-profit products to expensive, high-profit items (such as China moving from producing clothing to technology and cars), the GDP per Capita of these countries can rise dramatically.

If a country has higher than average GDP per Capita, we can also distinguish it into two groups. The first group is if the country has a lower than 65% labor force participation rate. If so, then the country has probably become high-income but has a highly dependent aging population (such as Europe or Japan). This means that the country’s economy is very efficient, as it is able to produce vast amounts of wealth despite not having many workers. The next group of countries is those with a labor force participation rate greater than 65%. This means that the country is in its economic prime. With a high labor force participation rate, this suggests that the country has a low number of dependents and a high number of workers engaged in high-skilled tasks. This is what leads to very high-income economies such as the United States.

I used a scatterplot as it is a strong choice for comparing two quantative variables for many different countries as I have done. In my case these variables are Labor Force Participation Rate (X-axis) and GDP per Capita (PPP) (Y-axis). A scatterplot also works well as it shows relationships and trends clearly. A scatterplot is also a great way to show the data for many countries as compared to bar plots (very skinny bars) the space representing each country (each dot on the plot) is very small allowing us to display data for all countries easily without sacrificing visual clarity. 

Through the observation of these different groups and the amount of dots in each group in the scatter plot, we can judge the state of the world economy. Ideally, the more countries that cross into the top right quadrant, the stronger the world economy is overall. In addition to this, the more economies which graduate from the bottom left quadrant, the better. As for colors, I decided to go with default color settings (blue in this case). This is because the x-axis and y-axis already encode for all variables we want to show (labor force and GDP per Capita). Furthermore, even though I could change the color for each country or continent, the purpose of the graph is to give a snapshot of the global economy and macro-economic priciples, not that of specific countries. This is why the default settings of one color was used.

In [ ]:
labor_2024 = df_labor_filtered[['Country Name', '2015']].dropna().copy()
labor_2024['2015'] = pd.to_numeric(labor_2024['2015'], errors='coerce')
top_20 = labor_2024.dropna().sort_values('2015', ascending=False).head(20)

plt.figure(figsize = (10,6))
plt.bar(top_20['Country Name'], top_20['2015'], color=plt.cm.viridis([x/20 for x in range(20)]))
        
plt.xticks(rotation=45, ha='right')
plt.xlabel('Country')
plt.ylabel('Labor Force Participation Rate (%)')
plt.title('Top 20 Countries by Labor Force Participation Rate (2015)')
plt.tight_layout()
plt.savefig("plot3.png")
plt.show()

This graph shows the top 20 countries by Labor Force Participation Rate in 2015. This bar plot is very important as it helps contextualize our earlier interactive graph showing the top 20 countries by GDP per Capita (PPP) in 2015. This bar plot serves as a way to take a look at and enforce our top 20 list from before by comparing and contrasting the two lists. Countries which are reprsented on the bar charts for both GDP per Capita and Labor Force Participation are countries which not only have a lot of wealth per individual (GDP per Capita) but that wealth is actual shared among all individuals as most people are actually working (high labour force participation rate).

This shows us that countries like Iceland, Qatar, Switzerland, Netherlands, Sweden and Norway are truly some of the strongest economies on earth as they are present in both of the bar charts we have created. This shows us that these are the most robust economies, something which either graph on its own can't do.

This is because if we just look at the first bar graph showing only countries with highest GDP per Capita (PPP), we see many rich countries such as Saudi Arabia, the United States, Bermuda and the Cayman Islands. The disclusion of these countries in this second bar graph on Labor Force Participation Rate shows us that these economies, despite being very large and efficient, do have problems such as inequality (Saudi Arabia and to a lesser extent the United States) or are inflated due to being tax shelters (Bermuda and the Cayman Islands). The disclusion of these countries from the second bar graph shows that despite these economies being highly developed and properous, their citizens may not be sharing these wealth and prosperity equally. We can contrast these with countries on both bar graphs such as Iceland, Norway, Switzerland, Netherlands where the inclusion on both bar graphs indicates that not only are these economies developed but they also provide all their citizens the opportunity to take part and benefit from the economy.

On the other hand, if we just look at the second bar graph showing only countries with highest Labour Force Participation Rate, we see many countries such as North Korea, Madagascar, Nigeria, Cambodia, Eritrea. The disclusion of these countries in this first bar graph on GDP per Capita (PPP) shows us that these economies, despite using all their human capital resources, are wildly inefficient as despite having all their citizens working, they can't product the same level of goods and services as other economies. This is because many of these economies are centrally planned (such as North Korea) which leads to artifically high number of labor force participation as workers are assigned jobs by the government (however these jobs are inefficient leading to the poor performance of the economy). Furthermore, only taking a look at this bar graph would also show us many countries where large percentages of the population work but in very inefficent industries (such as agriculture) such as Etheopia, Nigeria, Eritreia, Burkina Faso. These economies, despite having large sources of population willing and able to work are not able to capatilize on their vast human capital as it is tied up in inefficient endeavours such as subsistence farming.  We can contrast these with countries on both bar graphs such as Iceland, Norway, Switzerland, Netherlands where the inclusion on both bar graphs indicates that not only are these economies drawing from their large human capital resources but they also employ them in efficient industries (high level manufacturing, financial services, technology, specilized services) as opposed to inefficient industries such as subsistence agriculture.

For this graph, I used a bar graph as I am comparing a single numerical value (labor force participation rate) across a small, discrete set of categories (countries). A bar graph also provides a clear way to rank countries, works well for categorical data such as countries, and is easy to interpret exact differnces (difference in bar height). As for colors, I decided to use viridis to help further encode for Labor Force participation. This is becuase for countries very close such as Mozambique and Norway the colors provided by viridis can be used to distinguish differences quickly (lighter the color the lower the labor force participation).
